## Making trees work - Exercise
```In this exercise you will experience with Decision Trees and Random Forests. During this part you will explore the different features of them and will plot your results. Hence, whenever exploration tasks are marked with (*), know that you are asked to plot two graphs (on the same plot): the training score against the explored feature and the test score against it.```

```~Ittai Haran```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

```Read the dataset. In this dataset, you are provided over a hundred variables describing attributes of life insurance applicants. The task is to predict the "Response" variable.```

```the dataset can be found in: ```https://drive.google.com/open?id=1t_P64gM1M1_c2n4PvH7AZoELH2CNh6ui

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('insurance_fixed.csv')
X = df.drop(['Response'], axis = 1)
Y = df['Response']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size = 0.7, test_size = 0.3)

```We will start by using Decision trees. Use a simple DecisionTreeClassifier with default values to predict on your train and on your test. Evaluate the model using the accuracy metric, which you can find in sklearn.```

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

```Unfortunately, you are at overfit. Now let's try to get better. Try playing with the max depth of the tree, for``` $1\leq depth \leq25$ ```(*) (This means you are asked to plot some graphs, remember? :) )```

```Choose the optimal max_depth based on the graph you got.```

```Choose the best max_depth you found. Now try playing with min_samples_leaf. use the following values:
[1, 10, 100, 300,700, 1000]. Do it also with max_depth = 20. What can we learn from the graphs? Please answer the question ```$\ \underline{in\ another\ cell}$```.(*)```

```Decision Tree is a very nice algorithm, especially because it is very intuitive and explainable. We can even draw it!
Train a simple Decision Tree with max_depth = 3. Call it basic_tree and run the cell below. Examine the file tree.png you created.```

In [ ]:
from sklearn.tree import export_graphviz
export_graphviz(basic_tree, out_file = 'tree.dot', filled  = True,
                rounded = True, feature_names = df.columns)
!dot -Tpng tree.dot -o tree.png

NameError: name 'basic_tree' is not defined

```Look at the tree you got. What, would you say, are the most important features?
As you recall, we talked about feature importance in the lecture notes. Use the attribute feature_importance_ of your tree to get a list of the most important features.```

```We will now move to Random Forest. Repeat the exlporations tasks with a Random forest with 100 trees (max depth and min samples leaf). In addition, vary the number of trees between 10 and 400, while maintaining low max_depth (*) and the max_feature parameter, between 0.1 and 1 (*). Try explaining the graphs you see ```$\ \underline{in\ a\ different\ cell}$```. Use the flag n_jobs = -1 in your experiments to accelerate your computation time. Make sure to understand where your model is overfitted.```

In [ ]:
from sklearn.ensemble import RandomForestClassifier

```As you could see, at least one of your graphs turned out to be very noisy. Use K Fold cross validation to evalute your model more accurately. In K Fold cross validation we split our data into K segments, and for each ```$\ 1\leq i\leq K\ $``` we test our model on the i-th segment while training it using the others.```

In [ ]:
from sklearn.model_selection import KFold

def get_acc_on_folds(clf, X, Y, all_but, seg):
    clf.fit(X.iloc[all_but], Y.iloc[all_but])
    return accuracy_score(Y.iloc[all_but], clf.predict(X.iloc[all_but])), accuracy_score(Y.iloc[seg], clf.predict(X.iloc[seg]))

training_score = []
test_score = []
number_of_trees = [10,40,80,120,180,200,250,300,350,400]
for number in number_of_trees:
    k_fold = KFold(5)
    all_grades = list(map(lambda x: get_acc_on_folds(RandomForestClassifier(n_estimators=number, max_depth=7), X, Y, x[0], x[1]), k_fold.split(X,Y)))
    training_score.append(np.mean(list(map(lambda x: x[0], all_grades))))
    test_score.append(np.mean(list(map(lambda x: x[1], all_grades))))
plt.plot(number_of_trees, training_score,label='train')
plt.plot(number_of_trees, test_score,label='test')
plt.xlabel('number of trees')
plt.ylabel('score')
plt.legend()
plt.title('score againts number of trees in the forest')
plt.show()

```Use the Random Forest to surpass the best score you got using Decision Tree.```

In [ ]:
rfc = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
rfc.fit(X_train, Y_train)
print(accuracy_score(Y_test, rfc.predict(X_test)))


## Extra thinking on feature importance

```We talked about feature importance in the lecture notes. get the feature importance of each feature using a decision tree and using a random forest. Use in both cases the best hyper parameters you found so far. Discuss the differences between the answers``` $\underline{in\ a\ cell}$.

In [ ]:
dtc = DecisionTreeClassifier(max_depth=16)
dtc.fit(X_train, Y_train)
print(sorted(zip(X.columns, rfc.feature_importances_), key=lambda x: -x[1]))
print(sorted(zip(X.columns, dtc.feature_importances_), key=lambda x: -x[1]))

```We can define a concept of feature importance for linear regression: Suppose you have two features, ```$x_1$ ```and``` $x_2$. ```Suppose that you got a linear regression of the form```

$y = 100\cdot x_1 + 1\cdot x_2$

```What feature is more important? What if we have -100 instead of 100? Generalize this idea to any number of features. Train a linear regression on your data and get the feature importances.```

In [ ]:
from sklearn.linear_model import LinearRegression
linear_reg = LinearRegression(n_jobs=-1)
linear_reg.fit(X_train, Y_train)
sorted(zip(X.columns, linear_reg.coef_), key=lambda x: -abs(x[1]))


## Feature selection
```We will now try using the feature_importance we can get from our models to do a wise feature selection.```

Do the following:



*   ```Take the best Random Forest model you got at the last part.```
*   ```Select the top 20 feature with the greatest feature importance.```
*   ```Train a KNN model with n_neighbors = 8 (and n_jobs = -1). What is its accuracy?```
*   ```Train a KNN model using the 20 features you found, again with n_neighbors = 8 (and n_jobs = -1). What is its accuracy? What can we learn from it?```
*   ```Repeat the two last tasks, this time with the best Random Forest configuration you found. Can you explain the results?```
*   ```Draw a graph where the y axis is a feature_importance sum, and the x axis is the number of features you must take to get this sum. How can this graph help you to explain your results?```

In [ ]:
important_columns = [x[0] for x in sorted(zip(df.columns, rfc.feature_importances_), key = lambda x: -x[1])[:20]]

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knnc = KNeighborsClassifier(n_neighbors = 8, n_jobs=-1)
knnc.fit(X_train, Y_train)
print(accuracy_score(X_test, knnc.predict(X_test)))

knnc = KNeighborsClassifier(n_neighbors = 8, n_jobs=-1)
knnc.fit(X_train[important_columns], Y_train)
print(accuracy_score(Y_test, knnc.predict(X_test[important_columns])))

In [ ]:
random_forest = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
random_forest.fit(X_train, Y_train)
print(accuracy_score(Y_test, random_forest.predict(Y_test)))

random_forest_subset = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
random_forest_subset.fit(X_train[important_columns], Y_train)
print(accuracy_score(Y_test, random_forest_subset.predict(X_test[important_columns])))

```We will now implement a primitive reduction of Seffi's feature selection method (you are encouraged to ask him about it):```
- ```Take the best Random Forest model you found.```
- ```Create a dictionary that holds {feature: its importance}.```
- ```Transpose the df matrix, so we may think on the columns as 'samples' and vice versa.```
- ```Normalize each 'sample', so its length is 1.```
- ```Use KMeans with n_clusters = 20 on the 'samples'.```
- ```From each cluster take the feature with the highest feature importance.```

In [ ]:
random_forest = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
random_forest.fit(X_train, Y_train)
print(accuracy_score(Y_test, random_forest.predict(X_test)))

In [ ]:
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans

feature_importance_dict = dict(zip(df.columns, random_forest.feature_importances_))
df_transpose = df.transpose()

norma = Normalizer()
df_transpose_normalized = norma.fit_transform(df_transpose)

kmeans = KMeans(n_clusters=20, n_jobs=-1)
features_df = pd.DataFrame(zip(list(df_transpose.index), kmeans.fit_predict(df_transpose_normalized)),
                           columns = ['feature', 'group'])
features_df['grade'] = features_df['feature'].apply(lambda x: feature_importance_dict[x])
features_df['feature_grade'] = zip(features_df['feature'], features_df['grade'])
features_selected = list(features_df.groupby('group')\
                         .agg({'feature_grade':lambda x: max(x, key = lambda y: y[1])[0]})['feature_grade'])

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
clf = KNeighborsClassifier(n_neighbors = 8, n_jobs=-1)
clf.fit(X_train, Y_train)
print(accuracy_score(Y_test, clf.predict(X_test)))

clf = KNeighborsClassifier(n_neighbors = 8, n_jobs=-1)
clf.fit(X_train[important_columns], Y_train)
print(accuracy_score(Y_test, clf.predict(Y_test[important_columns])))

clf = KNeighborsClassifier(n_neighbors = 8, n_jobs=-1)
clf.fit(X_train[features_selected], Y_train)
print(accuracy_score(Y_test, clf.predict(X_test[features_selected])))

## Ensemble methods and stacking
```In this part we will explore the concept of model stacking: that is, training a model, the combining model, on the outputs of several other models. Hence, the stacking method has two steps: first we train our models, and than we train the combining model using the outputs of those models.```

```In the setting of stacking models it is very important to train the several models on one segment of the data and train the combining model on another segment. Hence, start by splitting the data to 3 segments: train_1 segment, 35% of the data, train_2 segment, 35% of the data, and test segment, the last 30% of the data.```

In [ ]:
df = pd.read_csv('insurance_fixed.csv')
segments_train_test = np.split(df.index.values, (len(df)*np.array([0.35,0.7])).astype(int))
segments = segments_train_test[:-1]
test_segment = segments_train_test[-1]

X_test = df.iloc[test_segment].drop(['Response'], axis = 1)
Y_test = df.iloc[test_segment]['Response']

```Our first experiment is as follows: train a random forest of simple decision trees (30 trees, max_depth = 3), using train_1. Use the estimators of the forest to create 30*8=240 features: for each estimator get the probabilities it gives for the target to belong to any of the classes. You can get the list of the estimators using RandomForestClassifier.estimators_ and have the probabilities mentioned using model.predict_proba.
Using the new features you got (and them only), train a logistic regression (LogisticRegression).
Compare between the accuracy of the first random forest (on the test segment) and the accuracy of the stacked models (again, on the test segment).```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

C:\Users\user\Anaconda3\lib\importlib\_bootstrap.py:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
  return f(*args, **kwds)


In [ ]:
model_1 = RandomForestClassifier(n_estimators=30, max_depth=3, max_features=0.8, n_jobs=-1)

X = df.iloc[segments[0]].drop(['Response'], axis = 1)
Y = df.iloc[segments[0]]['Response']
model_1.fit(X,Y)
print('model accuracy: ' +str(accuracy_score(Y_test, model_1.predict(X_test).astype(int))))
print('log loss: ' + str(log_loss(Y_test, model_1.predict_proba(X_test))))

X = df.iloc[segments[1]].drop(['Response'], axis = 1)
Y = df.iloc[segments[1]]['Response']
all_results = np.concatenate(list(map(lambda x: x.predict_proba(X), model_1.estimators_)), axis=-1)

model = LogisticRegression()
model.fit(all_results, Y)

all_results_test = np.concatenate(list(map(lambda x: x.predict_proba(X_test), model_1.estimators_)), axis=-1)
predictions = model.predict_proba(all_results_test)

print('model accuracy: ' + str(accuracy_score(Y_test, model.predict(all_results_test))))
print('log loss: ' + str(log_loss(Y_test, model.predict_proba(all_results_test))))

```We will conduct a similar experiment: create a set of at least 5 different models, of different kinds - use algorithms we talked about in the course. Stack them to get a better model. Compare the accuracies of the models to the accuracy of your stacked model.```

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegressionCV, LogisticRegression, RidgeClassifier, RidgeClassifierCV

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC

In [ ]:
model_1 = DecisionTreeClassifier(max_depth=7)
model_2 = KNeighborsClassifier(3, n_jobs=-1)
model_3 = KNeighborsClassifier(6, n_jobs=-1)
model_4 = AdaBoostClassifier()
model_5 = LogisticRegression()
model_6 = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
all_models = [model_1, model_2, model_3, model_4, model_5, model_6]

X = df.iloc[segments[0]].drop(['Response'], axis = 1)
Y = df.iloc[segments[0]]['Response']
all_models = list(map(lambda x: x.fit(X,Y), all_models))
all_accs = list(map(lambda x: accuracy_score(Y_test, x.predict(X_test)), all_models))
all_results_test = np.concatenate(list(map(lambda x: x.predict_proba(X_test), all_models)), axis=-1)

X_seg_1 = df.iloc[segments[1]].drop(['Response'], axis = 1)
Y_seg_1 = df.iloc[segments[1]]['Response']
all_results = np.concatenate(list(map(lambda x: x.predict_proba(X_seg_1), all_models)), axis=-1)

In [ ]:
model = RidgeClassifier()
model.fit(all_results, Y_seg_1)

predictions = model.predict(all_results_test)

print('all accuracies: ' + str(all_accs))
print('final accuracy: ' + str(accuracy_score(Y_test, predictions)))

```As we said earlier, it is very important use two different train segments. What happens if you use the same train segment in both steps of the stacked model? Note that you now use more data to train your models, and also your combining model. Do you get better results? Do it and explain your results ```$\underline{\ in\ a\ cell\ below.}$

In [ ]:
df = pd.read_csv('insurance_fixed.csv')
segments_train_test = np.split(df.index.values, (len(df)*np.array([0.7])).astype(int))
segments = segments_train_test[:-1]
segments = segments + segments
test_segment = segments_train_test[-1]

X_test = df.iloc[test_segment].drop(['Response'], axis = 1)
Y_test = df.iloc[test_segment]['Response']

model_1 = DecisionTreeClassifier(max_depth=7)
model_2 = KNeighborsClassifier(3, n_jobs=-1)
model_3 = KNeighborsClassifier(6, n_jobs=-1)
model_4 = AdaBoostClassifier()
model_5 = LogisticRegression()
model_6 = RandomForestClassifier(n_estimators=200, max_depth=16, max_features=0.8, n_jobs=-1)
all_models = [model_1, model_2, model_3, model_4, model_5, model_6]

X = df.iloc[segments[0]].drop(['Response'], axis = 1)
Y = df.iloc[segments[0]]['Response']
all_models = list(map(lambda x: x.fit(X,Y), all_models))
all_accs = list(map(lambda x: accuracy_score(Y_test, x.predict(X_test)), all_models))
all_results_test = np.concatenate(list(map(lambda x: x.predict_proba(X_test), all_models)), axis=-1)

X_seg_1 = df.iloc[segments[1]].drop(['Response'], axis = 1)
Y_seg_1 = df.iloc[segments[1]]['Response']
all_results = np.concatenate(list(map(lambda x: x.predict_proba(X_seg_1), all_models)), axis=-1)

model = RidgeClassifier()
model.fit(all_results, Y_seg_1)

predictions = model.predict(all_results_test)

print('all accuracies: ' + str(all_accs))
print('final accuracy: ' + str(accuracy_score(Y_test, predictions)))

## Extrapolation in Decision Trees

In [ ]:
from sklearn.tree import DecisionTreeRegressor

X = np.linspace(0.0, 5.0, 83).reshape(-1,1)
y = np.sin(X).ravel()
X_test = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]

`Try to approximate the sine function using a decision tree regressor.
Use the tree's depth to get a more accurate approximation of the sine function. Plot the results (try at least 3 differents depths).`

In [ ]:
# Fit regression model
regr_1 = DecisionTreeRegressor(max_depth=2)
regr_2 = DecisionTreeRegressor(max_depth=5)
regr_3 = DecisionTreeRegressor(max_depth=7)
regr_1.fit(X, y)
regr_2.fit(X, y)
regr_3.fit(X, y)
# Predict
y_1 = regr_1.predict(X_test)
y_2 = regr_2.predict(X_test)
y_3 = regr_3.predict(X_test)
# Plot the results
plt.figure()
plt.scatter(X, y, s=20, edgecolor="black", c="darkorange", label="data")
plt.plot(X_test, y_1, color="r", label="max_depth=2", linewidth=2)
plt.plot(X_test, y_2, color="g", label="max_depth=5", linewidth=2)
plt.plot(X_test, y_3, color="b", label="max_depth=7", linewidth=2)
plt.xlabel("data")
plt.ylabel("target")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()

`We now define a new test set, with values out of the training set.
Try the Decision Tree regressor on this new test set.
Plot the obtained results.`

In [ ]:
X_test_new = np.arange(0.0, 10.0, 0.1)[:, np.newaxis]
y_test_new = np.sin(X_test_new).ravel()
# Predict
y_1_new = regr_1.predict(X_test_new)
y_2_new = regr_2.predict(X_test_new)
y_3_new = regr_3.predict(X_test_new)
# Plot the results
plt.figure()
plt.scatter(X, y, s=20, edgecolor="black", c="darkorange", label="data")
plt.scatter(X_test_new, y_test_new, s=20, edgecolor="black", c="orange", label="data")
plt.plot(X_test_new, y_1_new, color="r", label="max_depth=2", linewidth=2)
plt.plot(X_test_new, y_2_new, color="g", label="max_depth=3", linewidth=2)
plt.plot(X_test_new, y_3_new, color="b", label="max_depth=15", linewidth=2)
# plt.plot(X_test_new, y_4_new, color="pink", label="linear regressor", linewidth=2)
plt.xlabel("data")
plt.ylabel("target")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()

`Can you explain this phenomenon? How does the training of the tree causes it?`

`What other model you know could overcome this situation?`

## Model explaining- LIMING things up
```We will try to explain complicated models using Decision Trees, and will get to know some concepts in the field.```

```We will use the MNIST data set.```

In [ ]:
df = pd.read_csv('data/MNIST_train.csv')
target = df['label']
df = df.drop('label', axis = 1)
df_train, df_test, target_train, target_test = train_test_split(df, target, train_size = 0.7, test_size = 0.3)

```Train a LGBMClassifier on the data set, using max_depth=10, n_estimators=250 (and, of course, n_jobs = -1). How good is your model?```

In [ ]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(max_depth=10, n_estimators=250, n_jobs=-1, verbose = 1)
lgb.fit(df_train, target_train)
print(accuracy_score(target_train, lgb.predict(df_train)))
print(accuracy_score(target_test, lgb.predict(df_test)))

```Read The short paper Model-Agnostic Interpretability of Machine Learning by Riberio et al, which you can find in the papers directory. We are about to try to make the LIME method proposed in the paper work, using Decision Tree. We would like to train a simple and interpretable model that predicts, locally, the predictions of our LGBMClassifier.
Create the targets (train and test) for the interpretable model, as described in the paper.```

```Create a function that gets a sample and returns a weight function. The weight function will get a sample (or multiple samples) and return the each sample's weight, using the following function: ``` y_weight $ = \frac{1}{|x-y|+1}$

```where x is the original sample and y is the sample that we would like to get its weight.
Note: we are about to use this function in order to define the "neighborhood" of a point - this definition is crucial in LIME.```

```Make sure your function is good: pick a sample and print the images of the 20 most weighted samples in the data set, compared to the sample you picked.```

```for every digit, do the following:```
- ```take a sample of it from the dataset.```
- ```Our interpretable model is going to be a decision tree with max_depth = 5. Create the model.```
- ```Train it using the weights derived from the sample picked and the predictions of the complex LGBMClassifier model.```
- ```Create an empty image, and paint it by the feature_importance of each pixel, which you can get from the interpretable model.```
- ```Paint the image. Can you learn something from the image?```

```What are the biggest problems in LIME? What makes it limited? Regard, in your answer, the way the concept of "locality" is defined.```

```The field of interpreting models might be very important, especially when working with complex models or with unsupervised learning. This part was a small taste of it, and you are encouraged to ``` **deepen** ``` (see what I did here? :) ) your knowledge about it.```